# NYC Yellow Taxi Trip Data - Exploratory Data Analysis (2023)

**Author:** [Your Name]  
**Goal:** Uncover insights from 2023 Yellow Taxi data to help optimize operations, maximize revenue, and improve passenger experience.

---

### Project Objectives
- Perform thorough data cleaning and preprocessing
- Analyze temporal patterns (hour, day, month)
- Conduct financial and tip analysis
- Perform geospatial analysis using taxi zones
- Derive actionable business insights and recommendations

### Tech Stack
Python • Pandas • NumPy • Matplotlib • Seaborn • GeoPandas

## 1. Import Libraries

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Optional - for geospatial analysis
# import geopandas as gpd

plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('viridis')

%matplotlib inline

print('Libraries imported successfully!')

## 2. Load Data

In the original project, 12 monthly Parquet files were sampled (5% of trips per hour per day) and combined.

For demonstration, we will load the provided sample file.  
Replace the path below with your full cleaned dataset when available.

In [ ]:
# Load the sample data (replace with your full cleaned parquet/csv)
df = pd.read_csv('../data/sample_yellow_taxi_2023.csv')

# Convert datetime columns
df['tpep_pickup_datetime'] = pd.to_datetime(df['tpep_pickup_datetime'])
df['tpep_dropoff_datetime'] = pd.to_datetime(df['tpep_dropoff_datetime'])

print(f'Dataset shape: {df.shape}')
df.head()

In [ ]:
df.info()

## 3. Data Cleaning (Summary of steps performed)

In the full project the following cleaning steps were applied:

1. Dropped duplicate/unnecessary columns (e.g. duplicate airport_fee)
2. Removed rows with negative monetary values
3. Handled missing values in `passenger_count`, `RatecodeID`, `congestion_surcharge`
4. Removed logical errors:
   - Zero/negative fare or total amount
   - Near-zero distance with very high fare
   - Zero distance + zero fare but different pickup/dropoff zones
   - Invalid payment_type = 0
5. Capped extreme outliers (top 0.1%) for trip_distance, fare_amount, tip_amount, total_amount
6. Removed trips with passenger_count > 6

The sample provided is already cleaned.

In [ ]:
# Quick check for any remaining issues
print('Missing values:')
print(df.isnull().sum())
print('\nBasic statistics:')
df[['trip_distance', 'fare_amount', 'tip_amount', 'total_amount', 'passenger_count']].describe()

## 4. Feature Engineering

In [ ]:
# Extract useful time features
df['pickup_hour'] = df['tpep_pickup_datetime'].dt.hour
df['pickup_day'] = df['tpep_pickup_datetime'].dt.day_name()
df['pickup_month'] = df['tpep_pickup_datetime'].dt.month
df['pickup_month_name'] = df['tpep_pickup_datetime'].dt.strftime('%b')
df['pickup_dayofweek'] = df['tpep_pickup_datetime'].dt.dayofweek  # Monday=0

# Trip duration in minutes
df['trip_duration_min'] = (df['tpep_dropoff_datetime'] - df['tpep_pickup_datetime']).dt.total_seconds() / 60

# Fare per mile
df['fare_per_mile'] = df['fare_amount'] / df['trip_distance'].replace(0, np.nan)

# Tip percentage
df['tip_percentage'] = (df['tip_amount'] / df['total_amount']) * 100

print('New features created successfully!')
df[['pickup_hour', 'pickup_day', 'trip_duration_min', 'fare_per_mile', 'tip_percentage']].head()

## 5. Exploratory Data Analysis

### 5.1 Temporal Analysis - Hourly Pickups

In [ ]:
plt.figure(figsize=(12, 5))
sns.countplot(data=df, x='pickup_hour', color='steelblue')
plt.title('Hourly Taxi Pickups Distribution', fontsize=14, fontweight='bold')
plt.xlabel('Hour of Day')
plt.ylabel('Number of Pickups')
plt.xticks(range(0, 24))
plt.tight_layout()
plt.show()

busiest_hour = df['pickup_hour'].value_counts().idxmax()
print(f'Busiest hour: {busiest_hour}:00')

### 5.2 Day of Week Analysis

In [ ]:
days_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']

plt.figure(figsize=(10, 5))
sns.countplot(data=df, x='pickup_day', order=days_order, palette='viridis')
plt.title('Taxi Pickups by Day of the Week', fontsize=14, fontweight='bold')
plt.xlabel('Day of Week')
plt.ylabel('Number of Pickups')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

### 5.3 Monthly Trends

In [ ]:
monthly = df.groupby(['pickup_month', 'pickup_month_name']).size().reset_index(name='count')
monthly = monthly.sort_values('pickup_month')

plt.figure(figsize=(10, 5))
plt.plot(monthly['pickup_month_name'], monthly['count'], marker='o', linewidth=2, color='darkorange')
plt.title('Monthly Pickup Trends', fontsize=14, fontweight='bold')
plt.xlabel('Month')
plt.ylabel('Number of Pickups')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### 5.4 Payment Type Distribution

In [ ]:
payment_labels = {1: 'Credit Card', 2: 'Cash', 3: 'No Charge', 4: 'Dispute'}
payment_counts = df['payment_type'].value_counts()

plt.figure(figsize=(7, 7))
plt.pie(payment_counts, labels=[payment_labels.get(i, i) for i in payment_counts.index],
        autopct='%1.1f%%', startangle=90, colors=sns.color_palette('Set2'))
plt.title('Payment Type Distribution', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

### 5.5 Trip Distance vs Fare Amount

In [ ]:
plt.figure(figsize=(10, 6))
sns.scatterplot(data=df.sample(min(3000, len(df))), x='trip_distance', y='fare_amount', alpha=0.5)
plt.title('Trip Distance vs Fare Amount', fontsize=14, fontweight='bold')
plt.xlabel('Trip Distance (miles)')
plt.ylabel('Fare Amount ($)')
plt.tight_layout()
plt.show()

corr = df[['trip_distance', 'fare_amount']].corr().iloc[0, 1]
print(f'Correlation between trip_distance and fare_amount: {corr:.3f}')

### 5.6 Average Fare per Mile by Hour

In [ ]:
fare_by_hour = df.groupby('pickup_hour')['fare_per_mile'].mean()

plt.figure(figsize=(12, 5))
fare_by_hour.plot(kind='line', marker='o', color='teal')
plt.title('Average Fare per Mile by Hour of Day', fontsize=14, fontweight='bold')
plt.xlabel('Hour of Day')
plt.ylabel('Avg Fare per Mile ($)')
plt.xticks(range(0, 24))
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### 5.7 Tip Percentage Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Tip % by hour
tip_by_hour = df.groupby('pickup_hour')['tip_percentage'].mean()
axes[0].plot(tip_by_hour.index, tip_by_hour.values, marker='o', color='purple')
axes[0].set_title('Average Tip % by Hour')
axes[0].set_xlabel('Hour of Day')
axes[0].set_ylabel('Avg Tip Percentage')
axes[0].set_xticks(range(0, 24))

# Tip % by passenger count
tip_by_pax = df.groupby('passenger_count')['tip_percentage'].mean()
axes[1].bar(tip_by_pax.index, tip_by_pax.values, color='coral')
axes[1].set_title('Average Tip % by Passenger Count')
axes[1].set_xlabel('Passenger Count')
axes[1].set_ylabel('Avg Tip Percentage')

plt.tight_layout()
plt.show()

## 6. Key Insights

### Temporal Patterns
- Peak demand typically occurs during evening hours (around 6–8 PM) and morning rush hours.
- Weekdays show stronger rush-hour patterns compared to weekends.
- Certain months (e.g., May, October) tend to generate higher activity and revenue.

### Financial Insights
- Strong positive correlation between trip distance and fare amount.
- Credit card is the dominant payment method (~75-80%).
- Fare per mile and tip behavior vary by time of day and trip characteristics.

### Operational Insights
- Identifying high-demand zones and slow routes can help optimize fleet allocation.
- Night-time operations still contribute meaningful revenue despite lower volume.

### Recommendations
1. **Dynamic Pricing**: Consider higher rates during peak hours and high-demand zones.
2. **Fleet Optimization**: Deploy more vehicles during weekday rush hours and key night zones.
3. **Tip Improvement**: Focus on short trips and cash payments where tip percentages are lower.
4. **Route Efficiency**: Investigate high-traffic but slow routes for potential improvements.

## 7. Next Steps / Full Analysis

This notebook uses a **sample dataset** for demonstration.

To reproduce the complete analysis from the original project:

1. Download all 12 monthly Parquet files from the [TLC Trip Record Data page](https://www.nyc.gov/site/tlc/about/tlc-trip-record-data.page)
2. Perform the stratified hourly sampling (5% per hour per day)
3. Apply the full data cleaning pipeline
4. Load the taxi zones shapefile and perform geospatial analysis with GeoPandas
5. Expand the analysis to include:
   - Slowest routes by average speed
   - Top pickup/dropoff zones
   - Night vs Day revenue share
   - Vendor comparison and distance-tier pricing
   - Detailed tip behavior analysis

---

**Thank you for reviewing this project!**  
Feel free to connect with me on LinkedIn or check out more of my work.